In [ ]:
import os

FORMAT_TXT = "/media/SSD/data/CVPR_workshop/test_release/prediction_files_format/prediction_files_format 2/ABAW_VA_test_set_sample.txt"
PRED_TXT = "/media/SSD/data/CVPR_workshop/test_result/predictions.txt"
OUT_TXT = "/media/SSD/data/CVPR_workshop/test_result/predictions2.txt"


def read_prediction_file(pred_path):
    """
    Read predictions.txt and return it as:
    {image_location: (valence, arousal)}
    """
    pred_dict = {}

    with open(pred_path, "r", encoding="utf-8") as f:
        lines = [line.strip() for line in f if line.strip()]

    start_idx = 0
    if lines and lines[0].lower().replace(" ", "") == "image_location,valence,arousal":
        start_idx = 1

    for line in lines[start_idx:]:
        parts = line.split(",")
        if len(parts) != 3:
            continue

        image_location = parts[0].strip()
        valence = parts[1].strip()
        arousal = parts[2].strip()

        pred_dict[image_location] = (valence, arousal)

    return pred_dict


def read_format_file(format_path):
    """
    Read only the image_location order from format.txt
    """
    image_locations = []

    with open(format_path, "r", encoding="utf-8") as f:
        lines = [line.strip() for line in f if line.strip()]

    start_idx = 0
    if lines and lines[0].lower().replace(" ", "") == "image_location,valence,arousal":
        start_idx = 1

    for line in lines[start_idx:]:
        parts = line.split(",")
        if len(parts) >= 1:
            image_location = parts[0].strip()
            image_locations.append(image_location)

    return image_locations


def fill_missing_with_nearest_values(format_list, pred_dict):
    """
    Generate results following the format file order.
    - If a prediction exists, use it
    - Otherwise, use the previous value
    - If no previous value exists, use the next value
    """
    n = len(format_list)
    result = [None] * n

    for i, img_loc in enumerate(format_list):
        if img_loc in pred_dict:
            result[i] = pred_dict[img_loc]

    last_seen = None
    for i in range(n):
        if result[i] is not None:
            last_seen = result[i]
        else:
            if last_seen is not None:
                result[i] = last_seen

    next_seen = None
    for i in range(n - 1, -1, -1):
        if result[i] is not None:
            next_seen = result[i]
        else:
            if next_seen is not None:
                result[i] = next_seen

    for i in range(n):
        if result[i] is None:
            result[i] = ("0.0", "0.0")

    return result


def save_result(out_path, format_list, filled_values):
    with open(out_path, "w", encoding="utf-8") as f:
        f.write("image_location,valence,arousal\n")
        for img_loc, (v, a) in zip(format_list, filled_values):
            f.write(f"{img_loc},{v},{a}\n")


def main():
    pred_dict = read_prediction_file(PRED_TXT)
    format_list = read_format_file(FORMAT_TXT)

    filled_values = fill_missing_with_nearest_values(format_list, pred_dict)
    save_result(OUT_TXT, format_list, filled_values)

    print(f"[DONE] saved to: {OUT_TXT}")
    print(f"format count      : {len(format_list)}")
    print(f"prediction count  : {len(pred_dict)}")


if __name__ == "__main__":
    main()